In [1]:
from qiskit import QuantumCircuit,Aer,execute

ImportError: cannot import name 'Aer' from 'qiskit' (/usr/local/lib/python3.12/dist-packages/qiskit/__init__.py)

In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

In [6]:
def deutsch_oracle(case: str) -> QuantumCircuit:
    """
    4種類のf(x)に対応するオラクル回路を生成
    q0:　入力ビット x
    q1: 補助ビット y => y XOR f(x)
    """
    oracle = QuantumCircuit(2, name= f"Oracle({case})")
    
    if case =="constant_0":
        #fx = 0 何もしない
        pass
    elif case =="constant_1":
        #補助ビットを常に反転
        oracle.x(1)
    elif case =="balanced_id":
        #f(x)=x => x=1の時だけ補助ビットを反転
        oracle.cx(0,1)
    elif case == "balanced_not":
        #f(x) = Not x => x=0の時だけ補助ビットを反転
        oracle.x(0)
        oracle.cx(0,1)
        oracle.x(0)
    else:
        raise ValueError("Invalid case")
    
    return oracle
     

In [7]:
def run_deutsch(case: str):
    # 量子ビット2つ、古典ビット1つ（q0の測定結果のみ記録）
    qc = QuantumCircuit(2, 1)
    
    # 1. 補助ビット q1 を |1> に初期化
    qc.x(1)
    qc.barrier()
    
    # 2. 両方に H ゲートを適用して |+>|-> を作成
    qc.h([0, 1])
    qc.barrier()
    
    # 3. オラクルを回路に組み込む
    oracle = deutsch_oracle(case)
    qc.compose(oracle, inplace=True)
    qc.barrier()
    
    # 4. 入力ビット q0 に H ゲートをかけて干渉させ、測定
    qc.h(0)
    qc.measure(0, 0)
    
    # シミュレータで実行
    simulator = AerSimulator()
    compiled_qc = transpile(qc, simulator)
    result = simulator.run(compiled_qc, shots=1024).result()
    counts = result.get_counts()
    
    print(f"=== {case} ===")
    print(qc.draw(output="text"))
    print("測定結果:", counts, "\n")

In [8]:
for c in ["constant_0", "constant_1", "balanced_id", "balanced_not"]:
    run_deutsch(c)

=== constant_0 ===
           ░ ┌───┐ ░  ░ ┌───┐┌─┐
q_0: ──────░─┤ H ├─░──░─┤ H ├┤M├
     ┌───┐ ░ ├───┤ ░  ░ └───┘└╥┘
q_1: ┤ X ├─░─┤ H ├─░──░───────╫─
     └───┘ ░ └───┘ ░  ░       ║ 
c: 1/═════════════════════════╩═
                              0 
測定結果: {'0': 1024} 

=== constant_1 ===
           ░ ┌───┐ ░       ░ ┌───┐┌─┐
q_0: ──────░─┤ H ├─░───────░─┤ H ├┤M├
     ┌───┐ ░ ├───┤ ░ ┌───┐ ░ └───┘└╥┘
q_1: ┤ X ├─░─┤ H ├─░─┤ X ├─░───────╫─
     └───┘ ░ └───┘ ░ └───┘ ░       ║ 
c: 1/══════════════════════════════╩═
                                   0 
測定結果: {'0': 1024} 

=== balanced_id ===
           ░ ┌───┐ ░       ░ ┌───┐┌─┐
q_0: ──────░─┤ H ├─░───■───░─┤ H ├┤M├
     ┌───┐ ░ ├───┤ ░ ┌─┴─┐ ░ └───┘└╥┘
q_1: ┤ X ├─░─┤ H ├─░─┤ X ├─░───────╫─
     └───┘ ░ └───┘ ░ └───┘ ░       ║ 
c: 1/══════════════════════════════╩═
                                   0 
測定結果: {'1': 1024} 

=== balanced_not ===
           ░ ┌───┐ ░ ┌───┐     ┌───┐ ░ ┌───┐┌─┐
q_0: ──────░─┤ H ├─░─┤ X ├──■──┤ X ├─░─┤ H ├┤M├
  